In [9]:
import pandas as pd
# Read JSON file
# df = pd.read_json("./jobs_output.json")
df=pd.read_json("./jobs_output_in_english.json")
# Display the first few rows
print(df.head())

   serial_no     Media site                               Company name  \
0        2.0   Fudosanworks                           Bandai Co., Ltd.   
1        3.0   Fudosanworks                           Bandai Co., Ltd.   
2        4.0   Fudosanworks                          KI Home Co., Ltd.   
3        5.0   Fudosanworks   Urban Planning and Development Co., Ltd.   
4        6.0   Fudosanworks                       Sanei Home Co., Ltd.   

   Company logo                                              Title  \
0           NaN   Real estate office work/Locally based/Free na...   
1           NaN   〖Real Estate Sales〗Leave work on time and hav...   
2           NaN   Room Advisor: Real estate sales job with no p...   
3           NaN  Real estate rental room advisor: A job where s...   
4           NaN   Room Advisor: Total concentration! Customer s...   

                                         Catchphrase      Salary type  \
0   Hair color and nails are OK, so you can expre...   Monthl

In [10]:
print(df.columns)

Index(['serial_no', 'Media site', 'Company name', 'Company logo', 'Title',
       'Catchphrase', 'Salary type', 'Salary', 'Salary details',
       'Employment Type', 'Job Industry', 'Job Category', 'Social insurance',
       'Job Benefits Details1', 'Job Benefits Details2',
       'Holidays & Leaves Details', 'Description', 'Requirements',
       'Requirements summary', 'Service Form', 'Working hours',
       'One day work details', 'Nearest Station', 'Nearest station access',
       'Selection flow', 'Recruiter message', 'Postal Code', 'Address details',
       'Google Maps URL', 'Trial period duration', 'Trial period details',
       'Trial period salary', 'Trial period working hours', 'image data',
       'Tag'],
      dtype='object')


In [11]:
print(len(df))
print(df.shape)

185
(185, 35)


In [13]:
first_row = df.iloc[0]
print(first_row['Salary'])

230000


In [14]:
print( df['Salary'].unique())

[230000 280000 240000 ' 300,000 to 450,000' '230,000 to 360,000' 1020
 ' 300,000 and up' ' 250,000 to 400,000' 270000 ' 233290～281580'
 ' 350,000 to 420,000' ' 235,000 to 450,000' '280,000 to 400,000' 250000
 265000 ' 270,000 to 400,000' ' 240,000 to 400,000' ' 420,000 to 570,000'
 ' 265,000 to 500,000' ' 280,000~' ' 250,000 to 350,000' ' 200,000~'
 ' 305,000 to 460,000 yen' ' 270,000 to 450,000 yen'
 ' 280,000 to 450,000 yen' ' 250,000 to 350,000 yen'
 ' 300,000 to 800,000 yen' ' 285,000 yen' ' 278,000 to 302,000 yen'
 ' 290,000 to 350,000 yen' ' 260,000 to 360,000 yen'
 ' 300,000 yen to 1,500,000 yen' ' 260,500 to 290,000 yen'
 ' 4,000,000 to 5,500,000 yen' ' 300,000 to 660,000 yen'
 ' 360,000 to 430,000 yen' ' 350,000 to 500,000 yen' ' From 290,000 yen'
 ' 300,000 yen' ' 380,000 to 500,000 yen' ' From 309,550 yen'
 ' 300,000 to 500,000 yen' ' 250,000 to 466,000 yen' ' 330,000 yen'
 ' 280,000 to 800,000 yen' ' 270,000 to 800,000 yen'
 ' 250,000 to 800,000 yen' ' 272,000 yen' ' 285,00

# salary splitting for japanese text only

In [8]:
def salary_splitting(salary):
    """
    Splits a salary string into min_salary and max_salary.
    - Single values go into min_salary, max_salary = None
    - Ranges are split on '～'
    - Handles commas and '円'
    """
    if salary is None:
        return (None, None)

    # Convert to string and clean
    salary = str(salary).replace(" ", "").replace("円", "").replace(",", "")

    # Check for range
    if '～' in salary:
        parts = salary.split('～')

        # If both values exist → proper range
        if parts[0] and len(parts) > 1 and parts[1]:
            try:
                min_salary = int(parts[0])
            except:
                min_salary = None
            try:
                max_salary = int(parts[1])
            except:
                max_salary = None
            return (min_salary, max_salary)

        # Otherwise → treat as single value (min only)
        try:
            min_salary = int(parts[0] or parts[1])
            return (min_salary, None)
        except:
            return (None, None)

    # Single number → only min_salary
    try:
        min_salary = int(salary)
        return (min_salary, None)
    except:
        return (None, None)

        
min_salary, max_salary = salary_splitting("305,000 to 460,000円")
print(f"min_salary===={min_salary} max_salary====={max_salary}")

min_salary, max_salary = salary_splitting("250000")
print(f"min_salary===={min_salary} max_salary====={max_salary}")

min_salary, max_salary = salary_splitting("300000～")
print(f"min_salary===={min_salary} max_salary====={max_salary}")

min_salary, max_salary = salary_splitting("300000～450000")
print(f"min_salary===={min_salary} max_salary====={max_salary}")


min_salary====None max_salary=====None
min_salary====250000 max_salary=====None
min_salary====300000 max_salary=====None
min_salary====300000 max_salary=====450000


# Salary_splitting function for both japanese and english text 

In [15]:
import re

def salary_splitting(salary):
    """
    General salary splitter.

    Returns:
        (min_salary, max_salary)

    Handles:
        - 300,000 to 450,000
        - 300,000～450,000
        - 280,000~
        - From 290,000 yen
        - 300,000 and up
        - 285,000 yen
        - 250000
        - None
    """

    if salary is None:
        return (None, None)

    # Convert to string & normalize
    salary_str = str(salary).lower()

    # Remove currency & unnecessary words
    salary_str = (
        salary_str
        .replace("yen", "")
        .replace("円", "")
        .replace(",", "")
        .strip()
    )

    # Extract all numbers
    numbers = re.findall(r"\d+", salary_str)
    numbers = [int(n) for n in numbers]

    if not numbers:
        return (None, None)

    # Detect range keywords
    range_keywords = ["to", "～", "~", "-", "–", "—"]

    # CASE 1 → Proper range
    if any(k in salary_str for k in range_keywords) and len(numbers) >= 2:
        return (numbers[0], numbers[1])

    # CASE 2 → From / and up / trailing ~
    if (
        "from" in salary_str
        or "and up" in salary_str
        or salary_str.endswith("~")
        or salary_str.endswith("～")
    ):
        return (numbers[0], None)

    # CASE 3 → Single value
    return (numbers[0], None)




tests = [
    "305,000 to 460,000円",
    "300,000 and up",
    "From 290,000 yen",
    "280,000~",
    "233290～281580",
    "285,000 yen",
    250000,
    None
]

for t in tests:
    print(t, "->", salary_splitting(t))


305,000 to 460,000円 -> (305000, 460000)
300,000 and up -> (300000, None)
From 290,000 yen -> (290000, None)
280,000~ -> (280000, None)
233290～281580 -> (233290, 281580)
285,000 yen -> (285000, None)
250000 -> (250000, None)
None -> (None, None)
